# Lesson 12 — CMA-ES as the Acquisition Optimiser

**Goal:** replace random candidate sampling with **CMA-ES** to find the EI maximum more reliably, especially in higher dimensions.

**Surrogate:** Deep Ensembles (Lesson 11) — provides μ and σ for EI.  
**Acquisition optimizer (new):** CMA-ES via the `cma` package.

The BO loop stays the same:
```
fit surrogate → maximise EI → evaluate f → update data → repeat
```
Only the *maximise EI* step changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm
import cma

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

## Benchmark functions

We use three functions of increasing dimension to show how the advantage of CMA-ES grows:

| Function | Dimension | Notes |
|---|---|---|
| Forrester | 1D | Classic BO benchmark |
| Branin | 2D | Three near-equal minima |
| Hartmann-6 | 6D | Standard high-d BO test |

In [ ]:
def forrester(x):
    x = np.asarray(x).ravel()
    v = (6*x - 2)**2 * np.sin(12*x - 4)
    return float(v[0]) if v.size == 1 else v

def branin_norm(x):
    x = np.asarray(x).ravel()
    x1, x2 = x[0]*15 - 5, x[1]*15
    return float((x2 - 5.1*x1**2/(4*np.pi**2) + 5*x1/np.pi - 6)**2
                 + 10*(1 - 1/(8*np.pi))*np.cos(x1) + 10)

def hartmann6(x):
    x = np.asarray(x).ravel()
    A = np.array([[10,3,17,3.5,1.7,8],[.05,10,17,.1,8,14],
                  [3,3.5,1.7,10,17,8],[17,8,.05,10,.1,14]], float)
    P = 1e-4*np.array([[1312,1696,5569,124,8283,5886],[2329,4135,8307,3736,1004,9991],
                        [2348,1451,3522,2883,3047,6650],[4047,8828,8732,5743,1091,381]], float)
    return float(-np.sum([1,1.2,3,3.2]*np.exp(-np.sum(A*(x-P)**2, axis=1))))

print("Forrester(0.76) ≈", round(forrester(0.76), 3), "  (min ≈ -6.02)")
print("Branin(0.12, 0.82) ≈", round(branin_norm([0.12, 0.82]), 3), "  (min ≈ 0.40)")
print("Hartmann6 at optimum ≈", round(hartmann6([0.201,0.150,0.477,0.275,0.311,0.657]), 3))

## Deep Ensemble surrogate

Same architecture as Lesson 11: 5 independent MLPs, each `64-64-1` with ReLU.  
Uncertainty = standard deviation across member outputs.

In [ ]:
class _MLP(nn.Module):
    def __init__(self, d=1, h=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, 1))
    def forward(self, x): return self.net(x)

class DeepEnsemble:
    def __init__(self, n=5, d=1, h=64):
        self.members = []
        for i in range(n):
            torch.manual_seed(i*100 + d*7)
            self.members.append(_MLP(d, h))

    def fit(self, X_t, y_t, epochs=1500, lr=1e-3):
        loss_fn = nn.MSELoss()
        for m in self.members:
            opt = optim.Adam(m.parameters(), lr=lr)
            for _ in range(epochs):
                m.train(); opt.zero_grad()
                loss_fn(m(X_t).squeeze(), y_t).backward()
                opt.step()

    def predict_np(self, X_np):
        x = torch.from_numpy(X_np.astype(np.float32))
        preds = []
        for m in self.members:
            m.eval()
            with torch.no_grad():
                preds.append(m(x).squeeze().numpy())
        p = np.array(preds)
        return p.mean(0), p.std(0)

def to_tensor(a): return torch.from_numpy(np.asarray(a, dtype=np.float32))

def standardise(X, y):
    Xm, Xs = X.mean(0), X.std(0)+1e-8
    ym, ys = float(y.mean()), float(y.std())+1e-8
    return (X-Xm)/Xs, (y-ym)/ys, Xm, Xs, ym, ys

def lhs(n, d, rg):
    pts = np.zeros((n, d))
    for j in range(d):
        pts[:,j] = (rg.permutation(n) + rg.uniform(size=n)) / n
    return pts.astype(np.float32)

def ei_fn(mu, std, f_best, xi=0.01):
    imp = f_best - xi - mu
    Z   = imp / (std + 1e-9)
    return np.maximum(imp*norm.cdf(Z) + std*norm.pdf(Z), 0.0)

print("Classes defined.")

## The two acquisition optimisers

**Random sampling** — draw N candidates uniformly, evaluate EI on all, pick the best.  
Simple and fast, but misses sharp EI peaks, especially in high dimensions.

**CMA-ES** — start from a random x₀, iteratively adapt a multivariate Gaussian to the EI landscape.  
Evaluates EI ~200 times but finds much better maxima in ≥2D.

In [ ]:
def build_neg_ei(ensemble, Xm, Xs, ym, ys, f_best, dim):
    def f(x_unit):
        x = np.asarray(x_unit).reshape(1, dim)
        x_std = (x - Xm) / Xs
        mu_s, std_s = ensemble.predict_np(x_std)
        mu  = float(mu_s[0]) * ys + ym
        std = float(std_s[0]) * ys
        return float(-ei_fn(np.array([mu]), np.array([std]), f_best)[0])
    return f

def acq_random(ensemble, Xm, Xs, ym, ys, f_best, dim, rg, n_cand=2000):
    Xc = rg.uniform(0, 1, (n_cand, dim)).astype(np.float32)
    mu_s, std_s = ensemble.predict_np((Xc - Xm) / Xs)
    mu, std = mu_s*ys + ym, std_s*ys
    ei_vals = ei_fn(mu, std, f_best)
    idx = int(np.argmax(ei_vals))
    return Xc[idx], float(ei_vals[idx])

def acq_cmaes(ensemble, Xm, Xs, ym, ys, f_best, dim, rg):
    neg_ei = build_neg_ei(ensemble, Xm, Xs, ym, ys, f_best, dim)
    x0   = rg.uniform(0.2, 0.8, dim)
    opts = {'bounds': [[0.0]*dim, [1.0]*dim], 'maxiter': 200,
            'tolx': 1e-5, 'tolfun': 1e-5, 'verbose': -9,
            'seed': int(rg.integers(0, 2**31))}
    es = cma.CMAEvolutionStrategy(x0, 0.3, opts)
    es.optimize(neg_ei)
    x_best = np.clip(es.result.xbest, 0, 1).astype(np.float32)
    return x_best, -float(es.result.fbest)

print("Acquisition optimisers defined.")

## Part 1 — CMA-ES mechanics on Forrester

We train the Deep Ensemble on 8 Forrester points, compute the EI landscape,
then run CMA-ES and watch it converge to the EI maximum.

The right panel shows **x_best per CMA-ES generation** — how quickly CMA-ES homes in.

In [ ]:
n_init_viz = 8
X_viz = rng.uniform(0, 1, (n_init_viz, 1)).astype(np.float32)
y_viz = np.array([forrester(x) for x in X_viz], dtype=np.float32)
X_std_v, y_std_v, Xm_v, Xs_v, ym_v, ys_v = standardise(X_viz, y_viz)

print("Training ensemble on 8 Forrester points...")
de_viz = DeepEnsemble(n=5, d=1)
de_viz.fit(to_tensor(X_std_v), to_tensor(y_std_v), epochs=2000)
print("Done.")

In [ ]:
xs      = np.linspace(0, 1, 300, dtype=np.float32).reshape(-1, 1)
mu_s, std_s = de_viz.predict_np((xs - Xm_v) / Xs_v)
mu_f    = mu_s  * ys_v + ym_v
std_f   = std_s * ys_v
f_best_v= float(y_viz.min())
ei_vals = ei_fn(mu_f, std_f, f_best_v)
y_true_f= (6*xs.ravel()-2)**2 * np.sin(12*xs.ravel()-4)

# CMA-ES with history
cma_history = []
neg_ei_viz  = build_neg_ei(de_viz, Xm_v, Xs_v, ym_v, ys_v, f_best_v, dim=1)
es = cma.CMAEvolutionStrategy([0.5], 0.3,
     {'bounds': [[0.0],[1.0]], 'maxiter': 80, 'verbose': -9,
      'tolx': 1e-6, 'tolfun': 1e-6, 'seed': 42})
while not es.stop():
    sols = es.ask()
    es.tell(sols, [neg_ei_viz(s) for s in sols])
    cma_history.append(float(np.clip(es.result.xbest, 0, 1)))

x_cma  = float(np.clip(es.result.xbest, 0, 1))
x_rand = float(xs[np.argmax(ei_vals)])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(xs, y_true_f, 'k--', lw=1.5, alpha=0.4, label='True f(x)')
axes[0].plot(xs, mu_f, color='steelblue', lw=2.5, label='Ensemble μ')
axes[0].fill_between(xs.ravel(), mu_f-2*std_f, mu_f+2*std_f,
                     alpha=0.2, color='steelblue', label='±2σ')
axes[0].scatter(X_viz.ravel(), y_viz, c='black', s=70, zorder=5)
axes[0].set_title('Deep Ensemble surrogate\n(8 training points)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].fill_between(xs.ravel(), 0, ei_vals, alpha=0.3, color='darkorange')
axes[1].plot(xs, ei_vals, color='darkorange', lw=2, label='EI(x)')
axes[1].axvline(x_rand, color='green', lw=2.5, ls='--', label=f'Random x={x_rand:.3f}')
axes[1].axvline(x_cma,  color='red',   lw=2.5, ls=':',  label=f'CMA-ES x={x_cma:.3f}')
axes[1].set_title('EI landscape — where each method queries next')
axes[1].set_xlabel('x'); axes[1].set_ylabel('EI(x)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

axes[2].plot(np.arange(len(cma_history)), cma_history, color='red',
             lw=2, marker='o', markersize=3)
axes[2].axhline(x_cma, color='red', ls='--', lw=1, alpha=0.5)
axes[2].set_title('CMA-ES convergence\nx_best per generation')
axes[2].set_xlabel('CMA-ES generation'); axes[2].set_ylabel('x_best')
axes[2].grid(True, alpha=0.3)

plt.suptitle('CMA-ES mechanics: maximising EI over the Deep Ensemble', fontsize=12)
plt.tight_layout()
plt.show()

## Part 2 — Why random sampling fails in high dimensions

In 1D, 2000 uniform candidates densely cover the unit interval.  
In 6D, 2000 candidates cover roughly `2000 / 10^6 = 0.2%` of the space.

CMA-ES adapts its search to the EI shape — it needs ~200 EI evaluations regardless of dimension.

In [ ]:
configs = [
    ('Forrester 1D', forrester,   1, 10),
    ('Branin 2D',    branin_norm, 2, 20),
    ('Hartmann 6D',  hartmann6,   6, 50),
]
n_reps  = 8
results = {}

for name, func, dim, n_pts in configs:
    print(f"  {name}...", flush=True)
    rg2 = np.random.default_rng(0)
    X   = lhs(n_pts, dim, rg2)
    y   = np.array([func(x) for x in X], dtype=np.float32)
    X_std, y_std, Xm, Xs, ym, ys = standardise(X, y)
    de  = DeepEnsemble(n=5, d=dim)
    de.fit(to_tensor(X_std), to_tensor(y_std), epochs=1000)
    f_best = float(y.min())
    rand_ei, cma_ei = [], []
    for rep in range(n_reps):
        rg3 = np.random.default_rng(rep + 50)
        _, r = acq_random(de, Xm, Xs, ym, ys, f_best, dim, rg3)
        _, c = acq_cmaes(de, Xm, Xs, ym, ys, f_best, dim, rg3)
        rand_ei.append(r); cma_ei.append(c)
    results[name] = {'rand': np.array(rand_ei), 'cma': np.array(cma_ei), 'dim': dim}

print("Done.")

In [ ]:
names  = [c[0] for c in configs]
x_pos  = np.arange(len(names))
w      = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(x_pos-w/2, [results[n]['rand'].mean() for n in names], w,
            label='Random (2000 cands)', color='green', alpha=0.8)
axes[0].bar(x_pos+w/2, [results[n]['cma'].mean()  for n in names], w,
            label='CMA-ES', color='red', alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'{n}\n(d={results[n]["dim"]})' for n in names])
axes[0].set_title(f'Mean EI found ({n_reps} reps)\nHigher = better acquisition')
axes[0].set_ylabel('EI'); axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

ratios = [results[n]['cma'].mean() / (results[n]['rand'].mean() + 1e-12) for n in names]
colors = ['green' if r < 1.1 else 'orange' if r < 2 else 'red' for r in ratios]
bars   = axes[1].bar(x_pos, ratios, color=colors, edgecolor='black', alpha=0.85)
axes[1].axhline(1.0, color='black', ls='--', lw=1.5, label='Random = 1×')
for bar, r in zip(bars, ratios):
    axes[1].text(bar.get_x()+bar.get_width()/2, r+0.02,
                 f'{r:.2f}×', ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'{n}\n(d={results[n]["dim"]})' for n in names])
axes[1].set_title('CMA-ES / Random EI ratio\n>1× = CMA-ES found better point')
axes[1].set_ylabel('Ratio'); axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Acquisition quality vs dimension — CMA-ES advantage grows with d', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — BO convergence: CMA-ES vs random on Forrester

A full BO loop: fit Deep Ensemble → optimise EI → evaluate → update.  
We run multiple seeds and compare convergence speed.

In [ ]:
N_INIT = 6
N_ITER = 20
N_SEEDS = 4

def run_bo(func, dim, seed, acq_method='cmaes', epochs=1000):
    rg2 = np.random.default_rng(seed)
    torch.manual_seed(seed)
    X = lhs(N_INIT, dim, rg2)
    y = np.array([func(x) for x in X], dtype=np.float32)
    best = [float(y.min())]
    for _ in range(N_ITER):
        X_std, y_std, Xm, Xs, ym, ys = standardise(X, y)
        de = DeepEnsemble(n=5, d=dim)
        de.fit(to_tensor(X_std), to_tensor(y_std), epochs=epochs)
        f_best = float(y.min())
        if acq_method == 'cmaes':
            x_next, _ = acq_cmaes(de, Xm, Xs, ym, ys, f_best, dim, rg2)
        else:
            x_next, _ = acq_random(de, Xm, Xs, ym, ys, f_best, dim, rg2)
        y_next = func(x_next)
        X = np.vstack([X, x_next.reshape(1, dim)])
        y = np.append(y, y_next)
        best.append(float(y.min()))
    return np.array(best)

print(f"Running BO on Forrester ({N_SEEDS} seeds × {N_ITER} iters)...")
forr_curves = {'cmaes': [], 'random': []}
for seed in range(N_SEEDS):
    print(f"  seed {seed+1}/{N_SEEDS}", flush=True)
    forr_curves['cmaes'].append(run_bo(forrester, 1, seed, 'cmaes'))
    forr_curves['random'].append(run_bo(forrester, 1, seed, 'random'))
forr_curves = {k: np.array(v) for k, v in forr_curves.items()}
print("Done.")

In [ ]:
F_STAR_FORR = forrester(np.array([0.7572]))
iters = np.arange(N_ITER + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for method, label, color in [('cmaes','Deep Ensemble + CMA-ES','red'),
                               ('random','Deep Ensemble + Random','green')]:
    c = forr_curves[method]
    m, s = c.mean(0), c.std(0)
    axes[0].plot(iters, m, color=color, lw=2.5, label=label)
    axes[0].fill_between(iters, m-s, m+s, alpha=0.2, color=color)
    gap = np.maximum(c - F_STAR_FORR, 1e-4)
    axes[1].semilogy(iters, gap.mean(0), color=color, lw=2.5, label=label)
    for g in gap:
        axes[1].semilogy(iters, g, color=color, lw=0.7, alpha=0.25)

axes[0].axhline(F_STAR_FORR, color='black', ls='--', lw=1.5,
                label=f'f* = {F_STAR_FORR:.3f}')
axes[0].set_title('Best f found — Forrester 1D')
axes[0].set_xlabel('BO iteration'); axes[0].set_ylabel('Best f(x)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Gap to optimum (log)')
axes[1].set_xlabel('BO iteration'); axes[1].set_ylabel('f_best − f*')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3, which='both')

plt.suptitle('Forrester BO convergence — CMA-ES vs random acquisition', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — BO convergence on Branin 2D and Hartmann-6

The CMA-ES advantage becomes clearer in 2D and dominant in 6D.

In [ ]:
print("Running BO on Branin 2D...")
br_curves = {'cmaes': [], 'random': []}
for seed in range(N_SEEDS):
    print(f"  seed {seed+1}/{N_SEEDS}", flush=True)
    br_curves['cmaes'].append(run_bo(branin_norm, 2, seed, 'cmaes'))
    br_curves['random'].append(run_bo(branin_norm, 2, seed, 'random'))
br_curves = {k: np.array(v) for k, v in br_curves.items()}

print("\nRunning BO on Hartmann-6...")
h6_curves = {'cmaes': [], 'random': []}
for seed in range(N_SEEDS):
    print(f"  seed {seed+1}/{N_SEEDS}", flush=True)
    h6_curves['cmaes'].append(run_bo(hartmann6, 6, seed, 'cmaes',  epochs=800))
    h6_curves['random'].append(run_bo(hartmann6, 6, seed, 'random', epochs=800))
h6_curves = {k: np.array(v) for k, v in h6_curves.items()}
print("Done.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (curves, fstar, title) in enumerate([
    (br_curves, 0.3979, 'Branin 2D'),
    (h6_curves, -3.3224, 'Hartmann-6 (6D)'),
]):
    for method, label, color in [('cmaes','CMA-ES','red'),('random','Random','green')]:
        c = curves[method]
        m, s = c.mean(0), c.std(0)
        axes[row,0].plot(iters, m, color=color, lw=2.5, label=f'Ensemble + {label}')
        axes[row,0].fill_between(iters, m-s, m+s, alpha=0.2, color=color)
        gap = np.maximum(c - fstar, 1e-4)
        axes[row,1].semilogy(iters, gap.mean(0), color=color, lw=2.5,
                             label=f'Ensemble + {label}')
        for g in gap:
            axes[row,1].semilogy(iters, g, color=color, lw=0.7, alpha=0.2)
    axes[row,0].axhline(fstar, color='black', ls='--', lw=1.5,
                        label=f'f* = {fstar:.4f}')
    axes[row,0].set_title(f'{title} — best f found')
    axes[row,0].set_xlabel('BO iteration'); axes[row,0].set_ylabel('Best f(x)')
    axes[row,0].legend(fontsize=8); axes[row,0].grid(True, alpha=0.3)
    axes[row,1].set_title(f'{title} — gap (log)')
    axes[row,1].set_xlabel('BO iteration'); axes[row,1].set_ylabel('gap')
    axes[row,1].legend(fontsize=8); axes[row,1].grid(True, alpha=0.3, which='both')

plt.suptitle('CMA-ES vs random acquisition — advantage grows with dimension', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| | 1D Forrester | 2D Branin | 6D Hartmann |
|---|---|---|---|
| Random sampling | OK | Moderate | Poor |
| CMA-ES | Same | Better | Clearly better |

**Key insight:** CMA-ES evaluates EI ~200 times but adapts its distribution — it finds peaks that 2000 uniform samples miss in higher dimensions.

**Next:** Lesson 13 — compare all three surrogates (GP, MC Dropout, Deep Ensembles) using CMA-ES acquisition.